# 03 — Classical baseline comparison and metric visualization

Reads the real MSD held-out reports produced by Notebook 02 / `scripts/train_msd.py`. This notebook does not retrain models or alter patient splits.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
ROOT = Path.cwd().resolve()
if not (ROOT / "outputs").exists(): ROOT = ROOT.parent
REPORTS = ROOT / "outputs/reports"
RESOLUTION = 240  # also evaluate 16 or 32 for the quantum-compatible low-resolution study
summary_path = REPORTS / f"msd_test_summary_{RESOLUTION}.csv"
per_slice_path = REPORTS / f"msd_test_per_slice_{RESOLUTION}.csv"
per_patient_path = REPORTS / f"msd_test_per_patient_{RESOLUTION}.csv"
if not summary_path.exists():
    raise FileNotFoundError("Run Notebook 02 or scripts/train_msd.py before this notebook.")
summary = pd.read_csv(summary_path)
per_slice = pd.read_csv(per_slice_path)
per_patient = pd.read_csv(per_patient_path) if per_patient_path.exists() else pd.DataFrame()
summary.head()

In [ ]:
metrics = [c for c in ["macro_dice_active_mean", "macro_iou_active_mean"] if c in summary.columns]
plot_df = summary[["method", *metrics]].set_index("method")
ax = plot_df.plot(kind="bar", figsize=(10,5), ylim=(0,1), title=f"Held-out active-region test metrics · {RESOLUTION}×{RESOLUTION}")
ax.set_ylabel("Score"); ax.grid(axis="y", alpha=.25); plt.xticks(rotation=0); plt.tight_layout()

In [ ]:
region_cols = [f"{region}_dice_mean" for region in ("WT", "TC", "ET") if f"{region}_dice_mean" in summary.columns]
summary[["method", *region_cols]].set_index("method").plot(kind="bar", figsize=(10,5), ylim=(0,1), title="Region-wise Dice")
plt.ylabel("Dice"); plt.xticks(rotation=0); plt.tight_layout()

In [ ]:
# Patient-level spread: each patient contributes once to this aggregation.
per_patient.groupby("method")["macro_dice_active"].agg(["count", "mean", "std", "median", "min", "max"]).sort_values("mean", ascending=False) if not per_patient.empty else per_slice.groupby("method")["macro_dice_active"].agg(["count", "mean", "std", "median", "min", "max"])

## Interpretation checklist

1. Compare methods on exactly the same held-out patient slices.
2. Report patient count and slice count separately.
3. Include mean and standard deviation, plus patient-wise distribution where possible.
4. HD95 is in pixels at the selected resized resolution in this implementation; it is not millimetres.
5. Do not claim clinical validity or quantum speedup from these experiments.